In [14]:
import pandas as pd

In [15]:
df = pd.read_csv('temp/demotermine_cleaned.csv', parse_dates=['date', 'edit_date'])

# only german

In [16]:
df = df[df['lang'] == 'de']

# min, max text length

In [17]:
df = df[(df['text_length'] >= 100) & (df['text_length'] <= 400)]

In [18]:
len(df)

12690

# select sample 

In [19]:
twenty_percent = int(len(df) / 100 * 20)

In [21]:
sample_df = df.sample(twenty_percent, random_state=42)

In [22]:
sample_df.head()

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,channel_name,channel_id,channel_description,message_id,from_id,via_bot_id,date,...,forwards,fwd_from,replies,reply_to,media,views,id,cleaned_text,text_length,lang
18616,22078,22078,24675,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,15100,NaN,NaN,2021-08-28 07:38:30,...,2.0,NaN,"MessageReplies(replies=0, replies_pts=290295, ...",NaN,MessageMediaWebPage(webpage=WebPage(id=8720606...,2801.0,Demotermine1250288610151002021-08-28 07:37:48,🐣 🎶 Äußert euch sachlich 28.08.2021 um 09 : 2...,262,de
16997,19929,19929,22186,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,25395,NaN,NaN,2021-11-06 16:33:19,...,23.0,NaN,"MessageReplies(replies=0, replies_pts=290292, ...",NaN,MessageMediaDocument(document=Document(id=5204...,2465.0,Demotermine1250288610253952021-11-06 16:33:43,"Versuch eines Gesprächs mit Stern TV , das von...",133,de
9032,10855,10855,12331,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,16404,NaN,NaN,2021-08-29 15:35:18,...,0.0,NaN,"MessageReplies(replies=0, replies_pts=290293, ...",NaN,MessageMediaPhoto(photo=Photo(id=5865183099078...,2190.0,Demotermine1250288610164042021-08-29 15:33:55,# be2808 13.31Demozug Teil3 # Demokratischer W...,166,de
17788,20935,20935,23398,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,24049,NaN,NaN,2021-10-30 04:09:13,...,2.0,NaN,"MessageReplies(replies=0, replies_pts=290292, ...",NaN,MessageMediaDocument(document=Document(id=5181...,1019.0,Demotermine1250288610240492021-10-30 04:08:39,ICONIC : ADVANCE AUSTRALIA FAIR - AUSTRALIAN A...,189,de
8815,10622,10622,12059,Demotermine,1250288610,🔊 Wir Demokraten wollen miteinander in Kontakt...,16708,NaN,NaN,2021-08-31 19:38:29,...,14.0,NaN,"MessageReplies(replies=1, replies_pts=290293, ...",NaN,MessageMediaPhoto(photo=Photo(id=5294374304693...,4564.0,Demotermine1250288610167082021-08-31 19:45:07,"CUXHAVEN geht spazieren . USER USER 05092021 ,...",103,de


In [25]:
sample_df.to_csv('temp/sample_data.csv')

## Identify similar messages via Levenshtein Distance

In [26]:
from itertools import combinations
import Levenshtein as lev

In [28]:
df = sample_df.copy()

In [29]:
lev_df = pd.DataFrame() 
new_df = pd.DataFrame(combinations(df['cleaned_text'],2), columns=["text_1", "text_2"]) 
new_df["lev_score"] = new_df.apply(lambda x: lev.ratio(x[0],x[1]), axis=1) 
lev_df = pd.concat([lev_df, new_df], axis=0).reset_index(drop=True)

In [30]:
# Get message ids
lev_df_ids = pd.DataFrame() 
new_df_ids = pd.DataFrame(combinations(df['message_id'],2), columns=["message_id_1", "message_id_2"]) 
lev_df_ids = pd.concat([lev_df_ids, new_df_ids], axis=0).reset_index(drop=True)

In [33]:
# combine lev results with message ids 
levs = pd.DataFrame()
levs = pd.concat([lev_df,lev_df_ids], axis=1)

In [35]:
levs.head()

,text_1,text_2,lev_score,message_id_1,message_id_2
0,🐣 🎶 Äußert euch sachlich 28.08.2021 um 09 : 2...,"Versuch eines Gesprächs mit Stern TV , das von...",0.372796,15100,25395
1,🐣 🎶 Äußert euch sachlich 28.08.2021 um 09 : 2...,# be2808 13.31Demozug Teil3 # Demokratischer W...,0.412993,15100,16404
2,🐣 🎶 Äußert euch sachlich 28.08.2021 um 09 : 2...,ICONIC : ADVANCE AUSTRALIA FAIR - AUSTRALIAN A...,0.251656,15100,24049
3,🐣 🎶 Äußert euch sachlich 28.08.2021 um 09 : 2...,"CUXHAVEN geht spazieren . USER USER 05092021 ,...",0.295082,15100,16708
4,🐣 🎶 Äußert euch sachlich 28.08.2021 um 09 : 2...,# Helvetia Rundgang Ascona 27.10.21 — Guido Bö...,0.301075,15100,23667


In [37]:
# get ids of messages over certain threshold
second_ids = levs[levs['lev_score'] >= 0.8].message_id_2.tolist()

In [40]:
len(set(second_ids))

642

In [44]:
ids_to_drop = list(set(second_ids))

In [48]:
result_df = df[~df.message_id.isin(ids_to_drop)]

In [49]:
len(result_df)

1896

In [50]:
result_df.to_csv('temp/data_selection.csv')

In [51]:
result_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1896 entries, 18616 to 6690
Data columns (total 22 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Unnamed: 0.2         1896 non-null   int64         
 1   Unnamed: 0.1         1896 non-null   int64         
 2   Unnamed: 0           1896 non-null   int64         
 3   channel_name         1896 non-null   object        
 4   channel_id           1896 non-null   int64         
 5   channel_description  1896 non-null   object        
 6   message_id           1896 non-null   int64         
 7   from_id              0 non-null      float64       
 8   via_bot_id           0 non-null      float64       
 9   date                 1896 non-null   datetime64[ns]
 10  edit_date            1824 non-null   datetime64[ns]
 11  text                 1896 non-null   object        
 12  forwards             1896 non-null   float64       
 13  fwd_from             121 non-

In [53]:
columns_to_keep = ['id', 'message_id', 'date', 'cleaned_text', 'lang']

In [55]:
result_df = result_df[columns_to_keep]

In [58]:
result_df.head(20)

,id,message_id,date,cleaned_text,lang
18616,Demotermine1250288610151002021-08-28 07:37:48,15100,2021-08-28 07:38:30,🐣 🎶 Äußert euch sachlich 28.08.2021 um 09 : 2...,de
16997,Demotermine1250288610253952021-11-06 16:33:43,25395,2021-11-06 16:33:19,"Versuch eines Gesprächs mit Stern TV , das von...",de
9032,Demotermine1250288610164042021-08-29 15:33:55,16404,2021-08-29 15:35:18,# be2808 13.31Demozug Teil3 # Demokratischer W...,de
17788,Demotermine1250288610240492021-10-30 04:08:39,24049,2021-10-30 04:09:13,ICONIC : ADVANCE AUSTRALIA FAIR - AUSTRALIAN A...,de
8815,Demotermine1250288610167082021-08-31 19:45:07,16708,2021-08-31 19:38:29,"CUXHAVEN geht spazieren . USER USER 05092021 ,...",de
17994,Demotermine1250288610236672021-10-28 08:39:51,23667,2021-10-28 08:40:17,# Helvetia Rundgang Ascona 27.10.21 — Guido Bö...,de
11918,Demotermine1250288610119712021-06-28 17:40:13,11971,2021-06-28 17:40:13,"Aliens , Bullen-Fatzen , Eskalation , Feiern b...",de
23437,Demotermine125028861022352020-07-31 12:43:03,2235,2020-07-31 12:39:41,Achtung ! Ganz wichtige Mitteilung für die Dem...,de
7512,Demotermine1250288610187112021-09-20 23:51:34,18711,2021-09-20 23:46:07,📆 Mahnwache / Versammlung unter freiem Himmel ...,de
8198,Demotermine1250288610176552021-09-10 14:40:45,17655,2021-09-10 14:40:04,"Osnabrück , Montagsdemo , 13.9.2021. Diesmal w...",de


In [57]:
# result_df.to_csv('temp/data_selection_final.csv')

In [59]:
texts = result_df.cleaned_text.values

In [62]:
for t in texts: 
    print(t, '\n\n')

🐣 🎶 Äußert euch sachlich 28.08.2021 um 09 : 25 Uhr Deine Meinung zum Reichstagsstürmchen 2020 ? Sag der Bild-Zeitung bitte auf Twitter sachlich Deine Meinung zu diesem Thema bitte : URL # lernetweeten # rausausderblase Alexander Ehrlich Raus auf die Straßen USER 


Versuch eines Gesprächs mit Stern TV , das von der Polizei unterbunden wird . USER Raus auf die Straßen USER 👉 Übersicht / Overview 👈 


# be2808 13.31Demozug Teil3 # Demokratischer Widerstand erreicht wider # Demoverbot Brunnenstraße — Osnabrücker Sommer der Aufklärung URL 2:17 Raus auf die Straßen USER 


ICONIC : ADVANCE AUSTRALIA FAIR - AUSTRALIAN ANTHEM [ LIVE VIOLIN ACOUSTIC VERSION ] MELBOURNE , AUSTRALIA 30.10.21 RISE UP MELBOURNE USER Raus auf die Straßen USER 👉 Übersicht / Overview 👈 


CUXHAVEN geht spazieren . USER USER 05092021 , 12092021 , 19092021 , 26092021 Raus auf die Straßen USER 


# Helvetia Rundgang Ascona 27.10.21 — Guido Böni URL 3:33 Raus auf die Straßen USER 👉 Übersicht / Overview 👈 


Aliens , 